# Code to construct amorphous membrane simulation box (requires manual crosslink placing) - Written by Joseph Thomas and Copilot

## import molecules for simulation and create sim box

In [1]:
import numpy as np
from ase.io import read, write
from ase.build import molecule, sort
from ase import Atoms
from ase.visualize import view
#import nglview
import sys
#from elecChemDev.build import attach_randomly_attempts

at = read('monomer.pdb') #monomer used to be inputted
for k in ['occupancy', 'bfactor', 'residuenames', 'atomtypes', 'residuenumbers']:
    del at.arrays[k]
at.center()
#rotating monomer
at.rotate(0,'y') 
at.rotate(-3,'z') 

# Settings for polymer
Nrep = 2 #Number of repeating monomers
Ny = 3 #Number of chains
Nz = 1 #Number of stacks of chains
dLx = 10 #Monomer length in x
dLy = 15 #Monomer length in y
dLz = 15 #Monomer length in z

at.cell = [dLx, dLy, dLz] #Constructing cell
sim = at.repeat((Nrep,Ny,Nz))
sim.pbc = [True, True, True]
sim.center()

view(sim)
#nglview.show_ase(sim)

<Popen: returncode: None args: ['/Users/w20025414/miniforge3/envs/mace_base/...>

In [2]:
sim.center()
sim = sort(sim)

write('check.xyz',sim) #Write .xyz file and add crosslinks

## Check for atom overlaps and write membrane only files

In [5]:
from ase.visualize import view
from ase.io import read,write

#Read crosslinked membrane back in 
sim = read('check.xyz')
sim.cell = [dLx*Nrep, Ny*dLy+6, Nz*dLz+10] 
sim.center()

#Check for duplicate coordinates
import numpy as np
frac = np.round(sim.get_scaled_positions(), 6) 
_, idx, counts = np.unique(frac, axis=0, return_index=True, return_counts=True)
dupes = np.where(counts > 1)[0]

print("Duplicate fractional coords:", dupes)
write('check.traj', sim)
view(sim)

Duplicate fractional coords: []


<Popen: returncode: None args: ['/Users/w20025414/miniforge3/envs/mace_base/...>

## Create membrane-solvent system

In [7]:
import ase
from ase.io import read, write
from ase.build import molecule
import numpy as np
from ase import Atoms
from ase.visualize import view

# ------------------------------------------------
# Load membrane and molecules
# ------------------------------------------------
sim = read("check.traj")
sim.center()
acid = read('H3PO4.pdb') #Reading in acid molecules
for k in ['occupancy', 'bfactor', 'residuenames', 'atomtypes', 'residuenumbers']:
    del acid.arrays[k]
acid.center()
water = molecule('H2O')

num_water = 21 #Number of water molecules to be added
num_acid = 36 #Number of acid molecules to be added
min_dist = 2.0 #Minimum allowed distance between solvent molecules

# Simulation box taken from your membrane system
box = sim.get_cell().lengths()
box = np.array(box)
membrane_z = box[2] / 2
membrane_thickness = 4.0

# Initialise final system with membrane
system = sim.copy()
system.set_pbc([True, True, True])

# Overlap check
def molecules_overlap(new_atoms, existing_atoms):
    if len(existing_atoms) == 0:
        return False
    new_pos = new_atoms.get_positions()
    old_pos = existing_atoms.get_positions()
    diff = new_pos[:, None, :] - old_pos[None, :, :]
    dists = np.linalg.norm(diff, axis=2)
    return np.any(dists < min_dist)

# Generate a grid of candidate positions,
def generate_grid_points(n_required):
    """
    Build a grid large enough to always yield at least n_required
    valid (non-membrane) points, then return all valid points
    so callers can pick greedily from a large pool.
    """
    # Estimate how much of the z-range is blocked by the membrane
    z_blocked = membrane_thickness / (box[2] - 4)   # fraction of usable z
    z_fraction = max(1 - z_blocked, 0.1)

    # Scale up n_side so that after membrane exclusion we still have enough
    n_side = int(np.ceil((n_required / z_fraction) ** (1/3))) + 2

    xs = np.linspace(2, box[0] - 2, n_side)
    ys = np.linspace(2, box[1] - 2, n_side)
    zs = np.linspace(2, box[2] - 2, n_side)

    points = []
    for x in xs:
        for y in ys:
            for z in zs:
                if membrane_z - membrane_thickness / 2 <= z <= membrane_z + membrane_thickness / 2:
                    continue
                points.append(np.array([x, y, z]))

    if len(points) < n_required:
        print(f"WARNING: Only {len(points)} grid points available but {n_required} requested. "
              f"Consider increasing the box size or reducing molecule counts.")

    return points   # return ALL valid points — placement loop picks greedily

# Place molecules with overlap check
def place_molecules(mol_template, n_required, candidate_points, label="molecule"):
    global system
    placed = 0
    for pos in candidate_points:
        if placed == n_required:
            break
        mol = mol_template.copy()
        com = mol.get_center_of_mass()
        mol.translate(pos - com)
        if not molecules_overlap(mol, system):
            system += mol
            placed += 1

    if placed < n_required:
        print(f"WARNING: Only placed {placed}/{n_required} {label} molecules "
              f"— {n_required - placed} could not be placed without overlap.")
    else:
        print(f"Successfully placed all {placed} {label} molecules.")

# Build a single shared pool of candidate points
total_needed = num_acid + num_water
all_candidates = generate_grid_points(total_needed)
rng = np.random.default_rng(42)
rng.shuffle(all_candidates)

place_molecules(acid,  num_acid,  all_candidates, label="H3PO4")
place_molecules(water, num_water, all_candidates, label="H2O")

# Output
print("Final total atoms in system:", len(system))
sim = sort(sim)
system.write("membrane_system.xyz")
view(system)

Successfully placed all 36 H3PO4 molecules.
Successfully placed all 21 H2O molecules.
Final total atoms in system: 647


<Popen: returncode: None args: ['/Users/w20025414/miniforge3/envs/mace_base/...>

### Sanity check for numbers of each element

In [8]:
from collections import Counter

counts = Counter(system.get_chemical_symbols())
print(counts)

Counter({'H': 246, 'O': 173, 'C': 164, 'P': 36, 'N': 28})


### Triple check for coordinate overlap

In [9]:
import numpy as np
frac = np.round(system.get_scaled_positions(), 6) 
_, idx, counts = np.unique(frac, axis=0, return_index=True, return_counts=True)
dupes = np.where(counts > 1)[0]

print("Duplicate fractional coords:", dupes)

Duplicate fractional coords: []
